# 00 ? Environment and protocol audit

**Objective:** Inspect the scientific protocol and all fourteen available experiments.

**Experiment:** Historical Flat versus Shared-Hard baseline reconstruction, seed 42.

**Config:** `configs/experiments/flat/efficientnet_b0.yaml`

**Inputs:** Frozen configs, audit documentation, manifest locks.

**Outputs:** Displayed environment, architecture/config inventory, audit.

**Mode:** Audit only; no training or image inference. Every input is loaded from disk; no other notebook's kernel state is required.

Use **Restart Kernel ? Run All**. Real training is disabled until the build handover is reviewed.


In [ ]:
from pathlib import Path
import sys, os
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "configs/protocol.yaml").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from IPython.display import display, Markdown
from src.config import load_config
DATA_ROOT = os.environ.get("SKIN_CANCER_DATA_ROOT")
print("Dataset root:", Path(DATA_ROOT).expanduser().resolve() if DATA_ROOT else "NOT CONFIGURED ? set SKIN_CANCER_DATA_ROOT before image verification/training")


In [ ]:
config = load_config("configs/experiments/flat/efficientnet_b0.yaml")
from src.reproducibility import environment
display(environment())

In [ ]:
import pandas as pd
paths = sorted((ROOT / "configs/experiments").glob("*/*.yaml"))
configs = [load_config(p) for p in paths]
display(pd.DataFrame([{k:c[k] for k in ("experiment_id","architecture","system_type","seed")} for c in configs]))
assert len(configs) == 14

## Historical details that affect reproduction

Shared-Hard trains three task heads on one encoder. Its four-class hard endpoint uses Stage 1/2; the auxiliary T-category head and three-task validation selection must remain. Flat MobileNetV3 retains its historical projection/Hardswish layer.

In [ ]:
display(Markdown((ROOT/"docs/HISTORICAL_PROTOCOL_AUDIT.md").read_text(encoding="utf-8")))
display(Markdown((ROOT/"docs/UNRESOLVED_PROTOCOL_ITEMS.md").read_text(encoding="utf-8")))

## Summary and next step

Review the status and metrics displayed above. Missing artifacts mean **not run**, never a successful reproduction. Generated artifacts are listed in the output cells; scientific results remain separate from historical reference values.

**Next:** `01_dataset_and_split_verification.ipynb`. Preserve completed run directories, prediction files and checkpoint backups before continuing.
